In [1]:
import argparse, csv, json, os, sys, base64 
from PIL import Image
from pathlib import Path
from io import BytesIO
from dotenv import load_dotenv
from openai import OpenAI
MODEL = "gpt-5"  

#api key
#using abs paths
load_dotenv("/Users/siddmalik/Desktop/semantic-robotic-tuning-main/.env")
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [2]:
def gif_to_base64(gif_filepath): #converts image or gif into utf-8 string
    try:
        with open(gif_filepath, "rb") as gif_file: 
            encoded_string = base64.b64encode(gif_file.read()) #first convert to b64encode
            return encoded_string.decode('utf-8')  # Then decode bytes to a UTF-8 string
    except FileNotFoundError:
        print(f"Error: The file '{gif_filepath}' was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [ ]:
message = input("Hi, how can I help you?") #get user message
#file = input("Do you need to add any file path?")
file = "yes" #did not require this setup so automatically provides 15 frames of gifs 2_2
if file == "0":
    messages = [{"role": "user", "content": message}]
else:
    messages=[ #user message in json format
            {"role": "user", "content": message}
        ]
    for i in range(1, 31): #all 30 frames in json format appended to messages array
        messages.append(
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",  # In Responses API, use "input_text"
                        "text": "Your text message goes here"
                    }
                ]
            })
chat = client.chat.completions.create( #send to api
        model=MODEL,
        messages=messages,
        temperature=1
    )
text = chat.choices[0].message.content #get text response
usage = chat.usage #gets tokens consumed
data = {
        "usage": {
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens,
            "cached_tokens": usage.prompt_tokens_details.cached_tokens #tokens saved by cache
        },
    }
file_path = "/Users/siddmalik/Desktop/semantic-robotic-tuning-main/"
file_bytes = None
if "base64" in text: #checks if api send any files, which would be decoded from b64 and saved in abs path
    try:
        file_bytes = base64.b64decode(text.strip())
        with open(file_path, "wb") as f:
            f.write(file_bytes)
        print(f"File saved")
    except Exception:
        pass
print(text)
print(data)
reply = input("Reply? 0 for NO") #in case of a reply from user
while True:
    if reply == "0":
        break
    else: #saves both answer and reply of user to message for cache application
        messages.append({"role": "assistant", "content": text})
        messages.append({"role": "user", "content": reply})
        chat = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=1
    )
    text = chat.choices[0].message.content
    usage = chat.usage
    data = {
            "usage": {
                "prompt_tokens": usage.prompt_tokens,
                "completion_tokens": usage.completion_tokens,
                "total_tokens": usage.total_tokens,
                "cached_tokens": usage.prompt_tokens_details.cached_tokens
            },
        }
    print(text)
    print(data)    
    reply = input("Reply? 0 for NO") #until 0

Hi, how can I help you? We have the gif frames of a walker animation and want the Bezier control points of the joints (if revolute, joint angle and if prismatic, distance). The joints are Stance Hip (Revolute), Stance Shin Extension (Prismatic), Stance Knee (Revolute), Stance Thigh Extension (Prismatic), Swing Hip (Revolute), Swing Thigh Extension (Prismatic), Swing Knee (Revolute), Swing Shin Extension (Prismatic). The coordinates of these joints are given in the gif frames (ignore the Joint_BaseRotY). Additional information about the robot structure can be found in the URDF File: <!-- AMBER 2D Model Description                                         --> <robot name="human_PF_adjustable">   <material name="red">     <color rgba="0 0 0.8 1"/>   </material>    <!-- Linkage Definitions -->   <link name="Torso">     <inertial>       <origin xyz="0 0 -0.28507"/>       <mass value="13"/>       <inertia ixx="0" ixy="0" ixz="0" iyy="0.1940" iyz="0" izz="0"/>     </inertial>   </link>   <link

I can compute the Bezier control points once I have the joint state (angle/extension) trajectories over one step. From the GIFs I can read the world-space positions of the joints, but images alone are not enough to recover every prismatic value (you need the end-point of the shin, i.e., the ankle/foot point). If you can provide the per‑frame coordinates as numbers (CSV or JSON) for the eight markers that appear in the legend of each frame, plus the bottom “foot/end-effector” point on each leg (or knee→foot distance), I will return the numeric control points immediately.

Below is the exact procedure I will apply (you can also run the included code with your data to reproduce the result).

What we need from each frame i (2D coordinates in meters):
- LeftHip Li.hip = (x,z), LeftThighExtension Li.thx, LeftKnee Li.knee, LeftShinExtension Li.shx
- RightHip Ri.hip, RightThighExtension Ri.thx, RightKnee Ri.knee, RightShinExtension Ri.shx
- Optional but required to identify shin prismatic valu